In [2]:
import pandas as pd

# Upload your files 
files = [
    'C:\\Users\\Riyad\\projects\\medical-assistand-ml\\Original_Dataset.csv',
    'C:\\Users\\Riyad\\projects\\medical-assistand-ml\\Symptom_Weights.csv',
    'C:\\Users\\Riyad\\projects\\medical-assistand-ml\\Disease_Description.csv',
    'C:\\Users\\Riyad\\projects\\medical-assistand-ml\\medicine.csv',
    'C:\\Users\\Riyad\\projects\\medical-assistand-ml\\Doctor_Specialist.csv',
]

for file in files:
    try:
        df = pd.read_csv(file)
        print(f"\n{'='*60}")
        print(f" File: {file}")
        print(f"{'='*60}")
        print(f"Shape: {df.shape}")
        print(f"Columns: {list(df.columns)}")
        print(f"\nFirst 3 rows:")
        print(df.head(3))
    except Exception as e:
        print(f"Error reading {file}: {e}")


 File: C:\Users\Riyad\projects\medical-assistand-ml\Original_Dataset.csv
Shape: (4920, 18)
Columns: ['Disease', 'Symptom_1', 'Symptom_2', 'Symptom_3', 'Symptom_4', 'Symptom_5', 'Symptom_6', 'Symptom_7', 'Symptom_8', 'Symptom_9', 'Symptom_10', 'Symptom_11', 'Symptom_12', 'Symptom_13', 'Symptom_14', 'Symptom_15', 'Symptom_16', 'Symptom_17']

First 3 rows:
            Disease   Symptom_1              Symptom_2              Symptom_3  \
0  Fungal infection     itching              skin_rash   nodal_skin_eruptions   
1  Fungal infection   skin_rash   nodal_skin_eruptions    dischromic _patches   
2  Fungal infection     itching   nodal_skin_eruptions    dischromic _patches   

              Symptom_4 Symptom_5 Symptom_6 Symptom_7 Symptom_8 Symptom_9  \
0   dischromic _patches       NaN       NaN       NaN       NaN       NaN   
1                   NaN       NaN       NaN       NaN       NaN       NaN   
2                   NaN       NaN       NaN       NaN       NaN       NaN   

  Symptom

In [3]:
!pip install scikit-learn pandas numpy joblib -q


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib
import warnings
warnings.filterwarnings('ignore')

print(" Libraries imported successfully!\n")

 Libraries imported successfully!



In [5]:
df_original = pd.read_csv('C:\\Users\\Riyad\\projects\\medical-assistand-ml\\Original_Dataset.csv')
df_symptom_weights = pd.read_csv('C:\\Users\\Riyad\\projects\\medical-assistand-ml\\Symptom_Weights.csv')
df_disease_desc = pd.read_csv( 'C:\\Users\\Riyad\\projects\\medical-assistand-ml\\Disease_Description.csv')
df_medicine = pd.read_csv('C:\\Users\\Riyad\\projects\\medical-assistand-ml\\medicine.csv')

print(f" Original Dataset: {df_original.shape}")
print(f" Symptom Weights: {df_symptom_weights.shape}")
print(f" Disease Description: {df_disease_desc.shape}")
print(f" Medicine Dataset: {df_medicine.shape}\n")

 Original Dataset: (4920, 18)
 Symptom Weights: (130, 2)
 Disease Description: (41, 2)
 Medicine Dataset: (21714, 10)



In [6]:
print(" Preprocessing data...")

# Get all unique symptoms from the dataset
symptom_columns = [col for col in df_original.columns if col.startswith('Symptom_')]
print(f" Found {len(symptom_columns)} symptom columns")

# Get all unique symptoms (excluding NaN)
all_symptoms = set()
for col in symptom_columns:
    symptoms = df_original[col].dropna().unique()
    all_symptoms.update(symptoms)

all_symptoms = sorted(list(all_symptoms))
print(f" Total unique symptoms: {len(all_symptoms)}\n")

# Create symptom to index mapping
symptom_to_index = {symptom: idx for idx, symptom in enumerate(all_symptoms)}

 Preprocessing data...
 Found 17 symptom columns
 Total unique symptoms: 131



In [8]:
print(" Creating feature matrix...")

def create_feature_vector(row, symptom_columns, symptom_to_index):
    """Convert symptoms to binary feature vector"""
    feature_vector = np.zeros(len(symptom_to_index))

    for col in symptom_columns:
        symptom = row[col]
        if pd.notna(symptom) and symptom in symptom_to_index:
            feature_vector[symptom_to_index[symptom]] = 1

    return feature_vector

# Create features (X) and labels (y)
X = []
y = []

for idx, row in df_original.iterrows():
    feature_vec = create_feature_vector(row, symptom_columns, symptom_to_index)
    X.append(feature_vec)
    y.append(row['Disease'])

    if (idx + 1) % 1000 == 0:
        print(f"Processed {idx + 1}/{len(df_original)} rows...")

X = np.array(X)
y = np.array(y)

print(f"\n Feature matrix shape: {X.shape}")
print(f" Labels shape: {y.shape}")
print(f" Number of unique diseases: {len(np.unique(y))}\n")

 Creating feature matrix...
Processed 1000/4920 rows...
Processed 2000/4920 rows...
Processed 3000/4920 rows...
Processed 4000/4920 rows...

 Feature matrix shape: (4920, 131)
 Labels shape: (4920,)
 Number of unique diseases: 41



In [9]:
print(" Encoding disease labels...")

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print(f" Encoded labels shape: {y_encoded.shape}\n")

 Encoding disease labels...
 Encoded labels shape: (4920,)



In [10]:
print(" Splitting dataset into train and test...")

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f" Training set: {X_train.shape}")
print(f" Test set: {X_test.shape}\n")

 Splitting dataset into train and test...
 Training set: (3936, 131)
 Test set: (984, 131)

